In [1]:
# Load API keys from .env file
from dotenv import load_dotenv
load_dotenv()
print("Environment loaded!")

Environment loaded!


In [2]:
# Load resort documents using pypdf directly - no PyTorch needed!
from pypdf import PdfReader
from langchain_core.documents import Document

# Load PDF
reader = PdfReader("../data/raw/Countryside_Resort.pdf")
pdf_docs = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text.strip():
        pdf_docs.append(Document(
            page_content=text,
            metadata={"source": "Countryside_Resort.pdf", "page": i}
        ))

# Load text file
with open("../data/raw/countryside_resort_info.txt", "r", encoding="utf-8") as f:
    txt_content = f.read()
txt_docs = [Document(page_content=txt_content, metadata={"source": "resort_info.txt"})]

# Combine
all_docs = pdf_docs + txt_docs
print(f"Loaded {len(all_docs)} document sections")

Loaded 10 document sections


In [3]:
# Split documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)
print(f"Split into {len(chunks)} chunks")

Split into 30 chunks


In [4]:
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "faiss-cpu"],
    capture_output=True, text=True
)
print(result.stdout[-500:])
print(result.stderr[-200:])


Using Python 3.11.3 environment at: C:\Users\Raheem Khan\Desktop\countryside-resort-bot\.venv
Checked 1 package in 96ms



In [5]:
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import time

load_dotenv(override=True)

hf_token = os.getenv("HF_TOKEN")
print(f"✅ Token: {hf_token[:8]}...")

# Embeddings via HuggingFace API (confirmed working earlier)
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=hf_token,
)

# Embed in small batches — FAISS, no DLL, no SQLite
print("Building vectorstore...")
vectorstore = None

for i in range(0, len(chunks), 5):
    batch = chunks[i:i+5]
    print(f"  Batch {i//5 + 1}: chunks {i}–{i+len(batch)-1}")
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embedding=embeddings)
    else:
        vectorstore.add_documents(batch)
    time.sleep(2)  # avoid HF rate limit

# Save to disk
vectorstore.save_local("../data/vectorstore")
print(f"✅ Vectorstore saved with {vectorstore.index.ntotal} chunks!")

✅ Token: hf_LyomQ...


c:\Users\Raheem Khan\Desktop\countryside-resort-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vectorstore...
  Batch 1: chunks 0–4
  Batch 2: chunks 5–9
  Batch 3: chunks 10–14
  Batch 4: chunks 15–19
  Batch 5: chunks 20–24
  Batch 6: chunks 25–29
✅ Vectorstore saved with 30 chunks!


In [6]:
# Cell 5 — Test RAG retrieval
from langchain_community.vectorstores import FAISS

# Load vectorstore from disk
vectorstore = FAISS.load_local(
    "../data/vectorstore",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

# Test questions
questions = [
    "What is the room price?",
    "What is the check-in time?",
    "Is there a restaurant at the resort?",
]

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

for question in questions:
    print(f"\n🔍 Question: {question}")
    print("-" * 50)
    results = retriever.invoke(question)
    for i, doc in enumerate(results, 1):
        print(f"  Chunk {i}: {doc.page_content[:200]}...")


🔍 Question: What is the room price?
--------------------------------------------------
  Chunk 1: Rate/ Per Night
15,000/-
Room Type
Deluxe / Queen Room
Mountain View Deluxe
Standard Large / Twin
Standard Large / Twin
Classic Double / Twin
Standard Room
Standard Room
15,000/-
10,000/-
10,000/-
10,...
  Chunk 2: WHAT EVERY ROOM INCLUDES
-------------------------
- Complimentary breakfast for 2 adults
- Free WiFi
- Pool access
- Orchard access
- Free parking

ROOM TYPES AND RATES (SEASON 2026 - PKR per room pe...
  Chunk 3: ENTRY ROOM:
- Standard Room (cozy, well-appointed, great value)
  Low season: 6,000 | Base: 7,500 | Peak: 9,000 | Eid: 11,000

SEASONS:
- Low season: March-April, October
- Base season: May-June, Sept...

🔍 Question: What is the check-in time?
--------------------------------------------------
  Chunk 1: ADDITIONAL CHARGES
------------------
- Extra adult breakfast: PKR 800
- Child breakfast (under 10): PKR 400
- Extra mattress: PKR 1,500
- Airport pickup/drop: PKR 1

In [8]:
from langchain.tools import tool

@tool
def search_resort_info(query: str) -> str:
    """Search Countryside Resort documents for information about 
    rooms, prices, activities, policies, check-in, amenities and facilities.
    Use this for any question about the resort itself."""
    
    # Get relevant chunks from our vectorstore
    results = retriever.invoke(query)
    
    # Combine all chunks into one answer
    combined = "\n\n".join([doc.page_content for doc in results])
    
    return combined

# Test it
print(search_resort_info.name)
print(search_resort_info.description)
print("\nTest result:")
print(search_resort_info.invoke("what is the cancellation policy?"))

search_resort_info
Search Countryside Resort documents for information about 
    rooms, prices, activities, policies, check-in, amenities and facilities.
    Use this for any question about the resort itself.

Test result:
Com pany Policies
W e take 100%  advance paym ent prior to arrival.
Cancellation and refund policy:
- 50%  refund if cancelled 10 days before arrival.
- No refund if cancelled less than 10 days of arrival.
If flight is cancelled on the day of arrival?
-Guest has to provide proof of ticket with nam e.
-If proof is provided, guest can use am ount
deposited with-in one fiscal year tore-book on
prevailing rates.
If there is landslide on the route on day of arrival?

COMPANY POLICIES
----------------
- 100% advance payment required prior to arrival
- Cancellation: 50% refund if cancelled 10 days before arrival
- No refund if cancelled less than 10 days before arrival
- Flight cancelled on arrival day: Guest must provide proof of ticket.
  Amount can be used within one fi